# 04 Inspect Classifier Model Comparison

Use this notebook after running `04_compare_classifier_models.py`. The script does the heavy/reproducible model comparison; this notebook reads the saved CSV files and makes the results easier to inspect.

## 1. Load Comparison Outputs

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

COMPARISON_DIR = Path(
    "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/"
    "political_corruption_pipeline/classifier_comparison"
)

best_results_path = COMPARISON_DIR / "best_model_results.csv"
threshold_results_path = COMPARISON_DIR / "all_threshold_results.csv"
country_results_path = COMPARISON_DIR / "country_results_for_best_silver_thresholds.csv"
predictions_path = COMPARISON_DIR / "validation_prediction_comparison.csv"

for path in [best_results_path, threshold_results_path, country_results_path, predictions_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. First run: python3 04_compare_classifier_models.py")

best_results = pd.read_csv(best_results_path)
threshold_results = pd.read_csv(threshold_results_path)
country_results = pd.read_csv(country_results_path)
predictions = pd.read_csv(predictions_path)

print(f"Loaded comparison outputs from: {COMPARISON_DIR}")
print(f"Best-result rows: {len(best_results):,}")
print(f"Threshold rows:    {len(threshold_results):,}")
print(f"Country rows:      {len(country_results):,}")
print(f"Prediction rows:   {len(predictions):,}")

## 2. Combine All Comparison Runs

This section automatically combines every comparison output folder matching `classifier_comparison*`, for example `classifier_comparison` and `classifier_comparison_heavy`. The rest of the notebook uses this combined result set by default.

In [ ]:
BASE_DIR = Path(
    "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/"
    "political_corruption_pipeline"
)

comparison_dirs = {
    path.name.replace("classifier_comparison", "comparison").strip("_") or "standard": path
    for path in sorted(BASE_DIR.glob("classifier_comparison*"))
    if path.is_dir()
}

if not comparison_dirs:
    raise FileNotFoundError(f"No classifier comparison folders found under {BASE_DIR}")

print("Comparison folders found:")
for run_name, folder in comparison_dirs.items():
    print(f"- {run_name}: {folder}")

best_frames = []
threshold_frames = []
country_frames = []

for run_name, folder in comparison_dirs.items():
    best_path = folder / "best_model_results.csv"
    threshold_path = folder / "all_threshold_results.csv"
    country_path = folder / "country_results_for_best_silver_thresholds.csv"

    if not best_path.exists():
        print(f"Skipping {run_name}; missing {best_path.name}")
        continue

    data = pd.read_csv(best_path)
    data["comparison_run"] = run_name
    best_frames.append(data)

    if threshold_path.exists():
        data = pd.read_csv(threshold_path)
        data["comparison_run"] = run_name
        threshold_frames.append(data)
    else:
        print(f"Missing threshold table for {run_name}: {threshold_path}")

    if country_path.exists():
        data = pd.read_csv(country_path)
        data["comparison_run"] = run_name
        country_frames.append(data)
    else:
        print(f"Missing country table for {run_name}: {country_path}")

inspection_best_results = pd.concat(best_frames, ignore_index=True)
inspection_best_results = inspection_best_results.drop_duplicates(
    subset=["model", "embedding_model", "label_source", "train_rows", "threshold"],
    keep="first",
)

inspection_threshold_results = (
    pd.concat(threshold_frames, ignore_index=True)
    if threshold_frames
    else pd.DataFrame()
)
inspection_country_results = (
    pd.concat(country_frames, ignore_index=True)
    if country_frames
    else pd.DataFrame()
)

print(f"\nLoaded {len(inspection_best_results):,} best-result rows across {len(best_frames)} comparison run(s).")

all_best_display = inspection_best_results[
    [
        "comparison_run",
        "model",
        "embedding_model",
        "label_source",
        "train_rows",
        "threshold",
        "accuracy",
        "political_precision",
        "political_recall",
        "political_f1",
        "macro_f1",
        "weighted_f1",
        "predicted_positive_rate",
    ]
].sort_values(
    ["political_f1", "political_recall", "political_precision"],
    ascending=False,
).reset_index(drop=True)

all_best_display


## 3. Best Model Table

Sort by political-corruption F1 first. Also check precision, recall, and predicted-positive rate before choosing a final classifier.

In [ ]:
display_columns = [
    "comparison_run",
    "model",
    "embedding_model",
    "label_source",
    "train_rows",
    "threshold",
    "accuracy",
    "political_precision",
    "political_recall",
    "political_f1",
    "macro_f1",
    "weighted_f1",
    "predicted_positive_rate",
]

best_display = (
    inspection_best_results[display_columns]
    .sort_values(["political_f1", "political_recall", "political_precision"], ascending=False)
    .reset_index(drop=True)
)

best_display


## 4. Recommended Final Candidate

This cell recommends the best `silver_combined` model across all discovered comparison runs. We prefer `silver_combined` because it uses all silver-label data; a single-batch model should only be chosen if it is clearly and substantively better.

In [ ]:
silver_combined = inspection_best_results[
    inspection_best_results["label_source"].eq("silver_combined")
].copy()
silver_combined = silver_combined.sort_values(
    ["political_f1", "political_recall", "political_precision"],
    ascending=False,
)

recommended = silver_combined.iloc[0]

print("Recommended combined-silver candidate")
print(f"Comparison run: {recommended.get('comparison_run', '')}")
print(f"Embedding model: {recommended['embedding_model']}")
print(f"Threshold:       {recommended['threshold']}")
print(f"Political F1:    {recommended['political_f1']:.3f}")
print(f"Precision:       {recommended['political_precision']:.3f}")
print(f"Recall:          {recommended['political_recall']:.3f}")
print(f"Positive rate:   {recommended['predicted_positive_rate']:.2%}")

silver_combined[display_columns]


## 5. Threshold Sweep For A Candidate

In [ ]:
SELECTED_EMBEDDING_MODEL = recommended["embedding_model"]
SELECTED_LABEL_SOURCE = recommended["label_source"]
SELECTED_COMPARISON_RUN = recommended.get("comparison_run", None)

candidate_thresholds = inspection_threshold_results[
    inspection_threshold_results["embedding_model"].fillna("").eq(str(SELECTED_EMBEDDING_MODEL))
    & inspection_threshold_results["label_source"].eq(SELECTED_LABEL_SOURCE)
].copy()

if SELECTED_COMPARISON_RUN is not None and "comparison_run" in candidate_thresholds.columns:
    candidate_thresholds = candidate_thresholds[
        candidate_thresholds["comparison_run"].eq(SELECTED_COMPARISON_RUN)
    ].copy()

candidate_thresholds[
    [
        "threshold",
        "accuracy",
        "political_precision",
        "political_recall",
        "political_f1",
        "macro_f1",
        "weighted_f1",
        "predicted_positive_rate",
    ]
]


In [ ]:
ax = candidate_thresholds.plot(
    x="threshold",
    y=["political_precision", "political_recall", "political_f1"],
    marker="o",
    figsize=(8, 4),
)
ax.set_ylim(0, 1)
ax.set_title(f"Threshold sweep: {SELECTED_EMBEDDING_MODEL} / {SELECTED_LABEL_SOURCE}")
ax.set_ylabel("Score")
ax.grid(True, alpha=0.3)

## 6. Country-Level Validation

In [ ]:
candidate_country = inspection_country_results[
    inspection_country_results["embedding_model"].fillna("").eq(str(SELECTED_EMBEDDING_MODEL))
    & inspection_country_results["label_source"].eq(SELECTED_LABEL_SOURCE)
].copy()

if SELECTED_COMPARISON_RUN is not None and "comparison_run" in candidate_country.columns:
    candidate_country = candidate_country[
        candidate_country["comparison_run"].eq(SELECTED_COMPARISON_RUN)
    ].copy()

candidate_country = candidate_country.sort_values("political_f1", ascending=False)

candidate_country[
    [
        "country",
        "n",
        "political_support",
        "accuracy",
        "political_precision",
        "political_recall",
        "political_f1",
        "predicted_positive_rate",
    ]
]


In [ ]:
ax = candidate_country.sort_values("political_f1").plot.barh(
    x="country",
    y="political_f1",
    figsize=(8, 5),
    legend=False,
)
ax.set_xlim(0, 1)
ax.set_title("Political-corruption F1 by country")
ax.set_xlabel("F1")
ax.grid(True, axis="x", alpha=0.3)

## 7. Final Scoring Command

After deciding on the final model and threshold, run full-corpus scoring from the terminal. Pull the latest repo first if you use an E5 model, because `05_train_final_classifier.py` must apply the same E5 text prefix as the comparison script.

In [ ]:
print("Suggested final scoring command:\n")
print(
    "TMPDIR=/home/akroon/data/1t_storage/tmp \\\n"
    "HF_HOME=/home/akroon/data/1t_storage/huggingface_cache \\\n"
    "TRANSFORMERS_CACHE=/home/akroon/data/1t_storage/huggingface_cache \\\n"
    "CUDA_VISIBLE_DEVICES=1 \\\n"
    "nohup python3 -u 05_train_final_classifier.py \\\n"
    f"  --embedding-model {recommended['embedding_model']} \\\n"
    f"  --threshold {recommended['threshold']} \\\n"
    "  --score-corpus \\\n"
    "  > silver_classifier_final_scoring.log 2>&1 &"
)
